<a href="https://colab.research.google.com/github/aditya01ad/colab_codes/blob/main/notebooks/OR_Prep/OR_prep.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Module 0 : Topic 1**

In [ ]:
!pip install pulp

In [ ]:
import sys
print(sys.version)

3.13.15 (main, Aug  6 2026, 11:06:23) [GCC 11.4.0]


In [ ]:
from pulp import *

In [ ]:
# Control flow - shipping cost
def shipping_cost(weight_kg, distance_km, base_rate=5.0):

    """
    Calculating . . .
    Args:
        weight_kg: Weight in kilogram
        distance_km: Distance in kilometer
        base_rate: Base rate per kg per km(default 5.0)
    Return:
        total cost
    """

    if weight_kg  <= 0 or distance_km <= 0:
      return 0.0

    cost = weight_kg * distance_km * base_rate

    #discount for bulk shopping
    if weight_kg > 100:
      cost = cost * 0.85 # 15%
    # surcharge for long distance
    if distance_km  > 1000:
      cost = cost * 1.10 #10%
    return round(cost, 2)


## Test
print(shipping_cost(50, 200))
print(shipping_cost(150, 200))
print(shipping_cost(50, 1500))
print(shipping_cost(-5, 100))
print(shipping_cost(200, 2000))

50000.0
127500.0
412500.0
0.0
1870000.0


In [ ]:
## List and loops - Processing multiple shipments

# List of shipments: [weight, distsnce]

shipments = [
    [45, 320],
    [120, 450],
    [80, 1200],
    [200, 90],
    [15, 60],
    [300, 2500],
    [0, 500],
    [75, -10]
]

# calculate for each shipment
shipment_costs = []
for i, shipment in enumerate(shipments):
  w, d = shipment
  cost = shipping_cost(w, d)
  shipment_costs.append([w, d, cost])
  print(f"Shipment {i+1}: {w}kg, {d}km -> ₹{cost}")
print("\nAll costs:", shipment_costs)

Shipment 1: 45kg, 320km -> ₹72000.0
Shipment 2: 120kg, 450km -> ₹229500.0
Shipment 3: 80kg, 1200km -> ₹528000.0
Shipment 4: 200kg, 90km -> ₹76500.0
Shipment 5: 15kg, 60km -> ₹4500.0
Shipment 6: 300kg, 2500km -> ₹3506250.0
Shipment 7: 0kg, 500km -> ₹0.0
Shipment 8: 75kg, -10km -> ₹0.0

All costs: [[45, 320, 72000.0], [120, 450, 229500.0], [80, 1200, 528000.0], [200, 90, 76500.0], [15, 60, 4500.0], [300, 2500, 3506250.0], [0, 500, 0.0], [75, -10, 0.0]]


In [ ]:
csv_data = """warehouse,store,distance_km,capacity_units
WH1,S1,210,500
WH1,S2,340,500
WH1,S3,180,500
WH1,S4,420,500
WH2,S1,390,600
WH2,S2,150,600
WH2,S3,460,600
WH2,S4,220,600
WH3,S1,310,400
WH3,S2,270,400
WH3,S3,510,400
WH3,S4,130,400
"""

# write to file
with open('shipping_data.csv', 'w') as f:
  f.write(csv_data)

# Read it back
with open('shipping_data.csv', 'r') as f:
  lines = f.readlines()
  print(lines)

['warehouse,store,distance_km,capacity_units\n', 'WH1,S1,210,500\n', 'WH1,S2,340,500\n', 'WH1,S3,180,500\n', 'WH1,S4,420,500\n', 'WH2,S1,390,600\n', 'WH2,S2,150,600\n', 'WH2,S3,460,600\n', 'WH2,S4,220,600\n', 'WH3,S1,310,400\n', 'WH3,S2,270,400\n', 'WH3,S3,510,400\n', 'WH3,S4,130,400\n']


In [ ]:
# Parse and compute
print("Shipping Data:")
print(f"{'Warehouse':12}{'Store':<8}{'Distance':>10}{'Cost(Rs)':>10}")
print("-" *43)

costs = []
for line in lines[1:]: #index 1 to last skip 0 only
  parts = line.strip().split(',')
  wh = parts[0]
  store = parts[1]
  dist = float(parts[2])
  cap = float(parts[3])

  cost = 100 + 2.5 *dist
  costs.append(cost)
  print(f"{wh:<12}{store:<8}{dist:>10.1f}{cost:>10.2f}")

print(f"\nTotal routes: {len(costs)}")
print(f"Avrage cost: Rs.{sum(costs)/len(costs):.2f}")

Shipping Data:
Warehouse   Store     Distance  Cost(Rs)
-------------------------------------------
WH1         S1           210.0    625.00
WH1         S2           340.0    950.00
WH1         S3           180.0    550.00
WH1         S4           420.0   1150.00
WH2         S1           390.0   1075.00
WH2         S2           150.0    475.00
WH2         S3           460.0   1250.00
WH2         S4           220.0    650.00
WH3         S1           310.0    875.00
WH3         S2           270.0    775.00
WH3         S3           510.0   1375.00
WH3         S4           130.0    425.00

Total routes: 12
Avrage cost: Rs.847.92


In [ ]:
route_details = [line.strip().split(',')[:2] for line in lines[1:]]
route_names = [f"{wh} to {store}" for wh, store in route_details]

# Find the minimum cost and its corresponding route using zip
min_cost_route1 = min(zip(costs, route_names))

print(f"Route with Minimum Cost: {min_cost_route1[1]}")
print(f"Cost: Rs.{min_cost_route1[0]:.2f}")

Route with Minimum Cost: WH3 to S4
Cost: Rs.425.00


In [ ]:
def load_shipping_data(filepath):
    """Load shipping data from CSV. Returns list of dicts."""
    data = []
    with open(filepath, 'r') as f:
        header = f.readline().strip()  # skip header
        for line in f:
            wh, store, dist, cap = line.strip().split(',')
            data.append({
                'warehouse': wh,
                'store': store,
                'distance_km': float(dist),
                'capacity': float(cap)
            })
    return data

def compute_route_cost(distance_km, fixed_cost=100, rate_per_km=2.5):
    """Compute shipping cost for a route."""
    return fixed_cost + rate_per_km * distance_km

def build_cost_matrix(data, warehouses, stores):
    """Build a 2D cost matrix from route data."""
    # Initialize with zeros
    matrix = [[0.0 for _ in stores] for _ in warehouses]

    for route in data:
        i = warehouses.index(route['warehouse'])
        j = stores.index(route['store'])
        matrix[i][j] = compute_route_cost(route['distance_km'])

    return matrix

# Use the functions
shipping_data = load_shipping_data('shipping_data.csv')
warehouse_list = sorted(list(set(r['warehouse'] for r in shipping_data)))
store_list = sorted(list(set(r['store'] for r in shipping_data)))

cost_mat = build_cost_matrix(shipping_data, warehouse_list, store_list)

print("Structured Cost Matrix:")
for i, wh in enumerate(warehouse_list):
    print(f"{wh}: {cost_mat[i]}")

Structured Cost Matrix:
WH1: [625.0, 950.0, 550.0, 1150.0]
WH2: [1075.0, 475.0, 1250.0, 650.0]
WH3: [875.0, 775.0, 1375.0, 425.0]


In [ ]:
print("Minimum cost for each store:")
print(f"{'Store':<8}{'Best Warehouse':<16}{'Min Cost':>10}")
print("-" * 34)

for j, store in enumerate(store_list):
    # Extract costs for this specific store from all warehouses
    store_costs = [cost_mat[i][j] for i in range(len(warehouse_list))]

    # Use zip to pair costs with warehouse names and find the minimum
    min_val, best_wh = min(zip(store_costs, warehouse_list))

    print(f"{store:<8}{best_wh:<16}{min_val:>10.2f}")

Minimum cost for each store:
Store   Best Warehouse    Min Cost
----------------------------------
S1      WH1                 625.00
S2      WH2                 475.00
S3      WH1                 550.00
S4      WH3                 425.00


# **Module 0 : Topic 2**

## **Basic array creation :-**

In [ ]:
import numpy as np
# I am writing code
# 1D array - like a list of stores, costs, etc
costs = np.array([12.5, 8.0, 15.3, 9.7])
print("1D costs:", costs)
print("Shape:", costs.shape)
print("Data type:", costs.dtype)

print("\n","-" *31)
# 2D array - a cost matrix (3 warehouses x 4 stores)
cost_matrix = np.array([
    [2.5, 3.0, 4.1, 2.8],
    [3.2, 2.9, 3.5, 4.0],
    [4.0, 3.8, 2.7, 3.1]
])

print("\nCost matrix:\n", cost_matrix)
print("Shape:", cost_matrix.shape)
print("Number of dimensions:", cost_matrix.ndim)

1D costs: [12.5  8.  15.3  9.7]
Shape: (4,)
Data type: float64

 -------------------------------

Cost matrix:
 [[2.5 3.  4.1 2.8]
 [3.2 2.9 3.5 4. ]
 [4.  3.8 2.7 3.1]]
Shape: (3, 4)
Number of dimensions: 2


## **Special arrays for OR model setup :-**

In [ ]:
# Zeros - initializing a deecision variable matrix (e.g. x[i, j] = 0 initially)
x = np.zeros((3, 4))
print("Zeros (decision vars):\n", x)

# Ones - for summation vectors
ones = np.ones(4)
print("\nOnes vector (4 stores):", ones)

# Identity - for constraints like x <= I * capacity
I = np.eye(3)
print("\nIdentity (3 wearhouses):\n", I)

# Range of values - capacities
capacities = np.arange(100, 500, 100)
print("\nCapacities:", capacities)

# Linsace - equally spaces demands
demands = np.linspace(50, 200, 15+1) # can change 15 o 150, 1500, 20000, etc
print("Demands:", demands)

Zeros (decision vars):
 [[0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]]

Ones vector (4 stores): [1. 1. 1. 1.]

Identity (3 wearhouses):
 [[1. 0. 0.]
 [0. 1. 0.]
 [0. 0. 1.]]

Capacities: [100 200 300 400]
Demands: [ 50.  60.  70.  80.  90. 100. 110. 120. 130. 140. 150. 160. 170. 180.
 190. 200.]


## **Array indexing and slicing — selecting routes :-**

In [ ]:
# Cost matrix
C = np.array([
    [2.5, 3.0, 4.1, 2.8],
    [3.2, 2.9, 3.5, 4.0],
    [4.0, 3.8, 2.7, 3.1]
])

# Get cost from warehouse 1 (index 0) to store 3 (index 2)
print("WH1 -> S3 cost:", C[0, 2])

# All costs from warehouse 2 (row index 1)
print("\nAll routes from WH2:", C[1, :])

# All costs to store 1 (column index 0)
print("\nAll costs to S1:", C[:, 0])

# Submatrix: first two warehouses, last two stores
sub = C[0:2, 2:4]
print("\nSubmatrix WH1-2 to S3-4:\n", sub)

# Boolean indexing: find routes with cost < 3.0
cheap = C < 3.0
print("\nCheap routes (cost < 3.0):\n", cheap)
print("Indices of cheap routes:", np.where(cheap))

WH1 -> S3 cost: 4.1

All routes from WH2: [3.2 2.9 3.5 4. ]

All costs to S1: [2.5 3.2 4. ]

Submatrix WH1-2 to S3-4:
 [[4.1 2.8]
 [3.5 4. ]]

Cheap routes (cost < 3.0):
 [[ True False False  True]
 [False  True False False]
 [False False  True False]]
Indices of cheap routes: (array([0, 0, 1, 2]), array([0, 3, 1, 2]))


## **Array operations — vectorized math:-**

In [ ]:
# per-unit cost matrix
unit_cost = np.array([[2.5, 3.0], [4.0, 2.8]])

# Fixed charge per route
fixed = np.array([[10.0, 15.0], [12.0, 10.0]])

# Total cost = unit_cost * quantity + fixed
quantity = 100
total = unit_cost * quantity + fixed
print("Total cost matrix:\n",total)

# Element-wise multi
discount_factors = np.array([[1.0, 0.9], [0.85, 1.0]])
discounted_cost = unit_cost * discount_factors
print("\nDiscount costs:\n", discounted_cost)

## Brodcasting : Add vector to each rows
base_per_wh = np.array([5.0, 7.0])
base_col = base_per_wh.reshape(-1,1)
total_with_base = unit_cost + base_col
print("\nUnit cost + wearhouse base fee:\n", total_with_base)


Total cost matrix:
 [[260. 315.]
 [412. 290.]]

Discount costs:
 [[2.5 2.7]
 [3.4 2.8]]

Unit cost + wearhouse base fee:
 [[ 7.5  8. ]
 [11.   9.8]]


## **Aggregations and statistics — summarizing model results :-**

In [ ]:
# Cost matrix
C = np.array([
    [2.5, 3.0, 4.1, 2.8],
    [3.2, 2.9, 3.5, 4.0],
    [4.0, 3.8, 2.7, 3.1]
])

# Total cost if one unit on every route
print("Total sum:", C.sum())
print("Mean cost per route:", C.mean())
print("Min cost: ", C.min(), " at index ", C.argmin())
print("Max cost: ", C.max(), " at index ", C.argmax())

# Row sums (total outbound cost per wearhouse)
print("\nPow sums (per wearhouse cost):", C.sum(axis=1))

# Column sum
print("Column sums (per store cost):", C.sum(axis=0))

# Standard deviation of costs - for risk analysis
print("\nStd dev of all routes costs:", C.std())


Total sum: 39.6
Mean cost per route: 3.3000000000000003
Min cost:  2.5  at index  0
Max cost:  4.1  at index  2

Pow sums (per wearhouse cost): [12.4 13.6 13.6]
Column sums (per store cost): [ 9.7  9.7 10.3  9.9]

Std dev of all routes costs: 0.5369667897862337


## **Linear algebra  :-**

In [ ]:
# Example: Solve a tiny transportation problem exactly (if total supply = total demand)
# This is just for illustration; normally use PuLP.

# Supply vector (2 warehouses)
supply = np.array([100, 150])
# Demand vector (3 stores)
demand = np.array([80, 120, 50])
# Cost matrix
C = np.array([[4, 6, 8], [5, 3, 7]])

# We'll just multiply a guessed shipment matrix:
x_guess = np.array([[50, 30, 20], [30, 90, 30]]) # satisfies supply/demand
total = np.sum(C * x_guess)
print("Total cost of guessd plan:", total)

# Matrix-vector product: Ax <= constraint form
A = np.array([[1, 2], [3, 4], [5, 6]])
x_vars = np.array([10, 20])
b = A @ x_vars # MAtrix multi
print("\nA @ x =", b)

# Solve linear system: 2x + 3y = 8, 4x + y = 6
coeff = np.array([[2, 3], [4, 1]])
rhs = np.array([8, 6])
sol = np.linalg.solve(coeff, rhs)
print("\nSolution x, y:", sol)

Total cost of guessd plan: 1170

A @ x = [ 50 110 170]

Solution x, y: [1. 2.]


## **Random number generation — creating test instances:**

In [ ]:
# Set seed for reproducibility
np.random.seed(123)

In [ ]:
# Set seed for reproducibility
np.random.seed(123)

# Random distances between50 and 500 km (3 warehouses, 4 stores)
distances = np.random.randint(50, 501, size=(3, 4))
print("Random distances:\n", distances)

# Random demands from normal distribution (mean=200, std=30)
demand = np.random.normal(loc=200, scale=30, size=4).astype(int)
# Ensure positive
demands = np.clip(demands, 50, None)
print("\nRandom demands:", demands)

# Random supply capacities drom uniform distribution
suplies = np.random.uniform(300, 600, size=3).round(2)
print("\nRandom suplies:", suplies)

# Cost = base * distance + noise
bsrate = 0.05
costs = bsrate * distances + np.random.normal(0, 2, size=distances.shape)
print("\nGenerated costs:\n", costs.round(2))

Random distances:
 [[415 432 372 148]
 [280  67 133 156]
 [173 107 264 275]]

Random demands: [ 50.  60.  70.  80.  90. 100. 110. 120. 130. 140. 150. 160. 170. 180.
 190. 200.]

Random suplies: [518.71 431.57 317.9 ]

Generated costs:
 [[23.73 20.32 17.71  6.53]
 [18.41  7.72  8.66  8.57]
 [10.12  8.33 11.33 16.1 ]]


## **Saving and loading arrays — bridging to CSV:**

In [ ]:
# Save the cost matrix to a file
np.savetxt('cost_matrix.csv', costs, delimiter=',', fmt='%.2f')
print("Saved to cost_matrix.csv")

# Load it back
loaded = np.loadtxt('cost_matrix.csv', delimiter=',')
print("\nLoaded matrix:\n", loaded)

# For multi-dimensional, use np.save / np.load (binary, faster)
np.save('distances.npy', distances)
loaded_dist = np.load('distances.npy')
print("\nLoaded distances:\n", loaded_dist)

Saved to cost_matrix.csv

Loaded matrix:
 [[23.73 20.32 17.71  6.53]
 [18.41  7.72  8.66  8.57]
 [10.12  8.33 11.33 16.1 ]]

Loaded distances:
 [[415 432 372 148]
 [280  67 133 156]
 [173 107 264 275]]


# **Module 0 — Topic 3: Pandas**

## Import and create a DataFrame from scratch:

In [1]:
import pandas as pd
import numpy as np

# Create a sample order dataset
orders = pd.DataFrame({
    'order_id': [101, 102, 103, 104, 105],
    'warehouse': ['WH1', 'WH2', 'WH1', 'WH3', 'WH2'],
    'store': ['S1', 'S2', 'S3', 'S1', 'S4'],
    'units': [200, 150, 300, 100, 250],
    'distance_km': [210, 340, 180, 420, 150]
})

print(orders)

   order_id warehouse store  units  distance_km
0       101       WH1    S1    200          210
1       102       WH2    S2    150          340
2       103       WH1    S3    300          180
3       104       WH3    S1    100          420
4       105       WH2    S4    250          150


## Loading from CSV and quick inspection:

In [2]:
# Create a CSV in the same way as before
csv_data = """order_id,warehouse,store,units,distance_km
101,WH1,S1,200,210
102,WH2,S2,150,340
103,WH1,S3,300,180
104,WH3,S1,100,420
105,WH2,S4,250,150
106,WH1,S2,180,260
107,WH3,S3,220,310
108,WH2,S1,90,390
"""
with open('orders.csv', 'w') as f:
    f.write(csv_data)

# Load
df = pd.read_csv('orders.csv')

# Inspect
print(df.head(3))
print("\nShape:", df.shape)
print("\nColumns:", df.columns.tolist())
print("\nData types:\n", df.dtypes)
print("\nMissing values:\n", df.isnull().sum())

   order_id warehouse store  units  distance_km
0       101       WH1    S1    200          210
1       102       WH2    S2    150          340
2       103       WH1    S3    300          180

Shape: (8, 5)

Columns: ['order_id', 'warehouse', 'store', 'units', 'distance_km']

Data types:
 order_id        int64
warehouse      object
store          object
units           int64
distance_km     int64
dtype: object

Missing values:
 order_id       0
warehouse      0
store          0
units          0
distance_km    0
dtype: int64


## Filtering and selecting — queries:

In [3]:
# Orders from WH1
wh1_orders = df[df['warehouse'] == 'WH1']
print("WH1 orders:\n", wh1_orders)

# Orders with units > 150 and distance < 300
big_close = df[(df['units'] > 150) & (df['distance_km'] < 300)]
print("\nBig and close orders:\n", big_close)

# Select only needed columns
df_small = df[['warehouse', 'store', 'units']]
print("\nSmall view:\n", df_small)

# Using .loc for label-based access
print("\nRow 2 (order 103):\n", df.loc[2])
print("\nFirst 2 rows, columns warehouse and units:\n", df.loc[:1, ['warehouse', 'units']])

# Using .iloc for position-based
print("\nThird row:\n", df.iloc[2])

WH1 orders:
    order_id warehouse store  units  distance_km
0       101       WH1    S1    200          210
2       103       WH1    S3    300          180
5       106       WH1    S2    180          260

Big and close orders:
    order_id warehouse store  units  distance_km
0       101       WH1    S1    200          210
2       103       WH1    S3    300          180
4       105       WH2    S4    250          150
5       106       WH1    S2    180          260

Small view:
   warehouse store  units
0       WH1    S1    200
1       WH2    S2    150
2       WH1    S3    300
3       WH3    S1    100
4       WH2    S4    250
5       WH1    S2    180
6       WH3    S3    220
7       WH2    S1     90

Row 2 (order 103):
 order_id       103
warehouse      WH1
store           S3
units          300
distance_km    180
Name: 2, dtype: object

First 2 rows, columns warehouse and units:
   warehouse  units
0       WH1    200
1       WH2    150

Third row:
 order_id       103
warehouse      WH1


## Aggregation and groupby — summarizing for OR models: